In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "12"
os.environ["MKL_NUM_THREADS"] = "12"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import platform, re, warnings, unicodedata, shutil
from datetime import datetime
from typing import List, Tuple, Optional, Sequence, Any

import torch
import pandas as pd
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
transformers.logging.set_verbosity_error()
torch.set_num_threads(12)
torch.set_num_interop_threads(3)

from docx import Document
from striprtf.striprtf import rtf_to_text

# --- RAG (sekundár) ---
from langchain_chroma import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

# ============ CONFIG ============

MODE              = "title"     # 'title' alebo 'keyword'
INPUT_DIR         = "./Input"
OUTPUT_DIR        = "./Output"

THESAURUS_XLSX    = "./Thesaurus/SK_Local_Thesaurus.xlsx"
THESAURUS_COL     = "slovak_terms"
STOPWORDS_TXT     = "./Input/stopword_dictionary.txt"

#PRIMARY_MODEL_ID   = "marcuscedricridia/8B-Nemotaur-IT"
SECONDARY_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
#FALLBACK_MODEL_ID  = "google/gemma-2-2b-it"

ENABLE_RAG_PRIMARY   = True
ENABLE_RAG_SECONDARY = True
ENABLE_RAG_FALLBACK  = True

MAX_NEW_TOKENS      = 96
BATCH_SIZE          = 4
DO_SAMPLE           = True
TEMPERATURE         = 0.2
TOP_P               = 0.9
REPETITION_PENALTY  = 2.0
MAX_TEXT_CHARS      = 8000

# Reasoning (self-consistency sampling)
REASONING_FOR_PRIMARY   = True
REASONING_FOR_SECONDARY = True
REASONING_FOR_FALLBACK  = False
REASONING_SAMPLES       = 3       
REASONING_TEMP          = 0.6
REASONING_TOP_P         = 0.9

# RAG: embedding + store
RAG_PERSIST_DIR    = "./RAG_store"
RAG_COLLECTION     = "sk_legal_local"
RAG_EMB_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"  # 384-d
RAG_TOP_K          = 5

PENALIZE_CZ = True
CZ_REGEX = re.compile(r"\b(řízení|ustanovení|přestupk|odpovědnost|veřejn|úřad|povinnost|prostředk|soud|správní|nemovitost|podání)\b", re.IGNORECASE)

# ============ UTIL ============

def configure_cuda_allocator():
    parts = ["max_split_size_mb:128"]
    if platform.system() == "Linux":
        parts.insert(0, "expandable_segments:True")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ",".join(parts)

configure_cuda_allocator()
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")

def load_stopwords(path=STOPWORDS_TXT):
    try:
        with open(path, "r", encoding="utf-8") as f:
            words = {line.strip().lower() for line in f if line.strip()}
        print(f"[STOP] Loaded {len(words)} stopwords")
        return words
    except FileNotFoundError:
        print("[STOP] No stopwords file; continuing.")
        return set()

STOPWORDS = load_stopwords()

def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    return term.split(' - ')[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r'^(.*?)(\s*\(([^)]+)\))\s*$', term)
    if not m:
        return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r'[\/,;]', inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r'\s+', r'\\s+', re.escape(s))
    return s.replace(r'\-', r'[-–—]')

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)

    pats = [re.compile(r'\b' + _wildify(base) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r'\b' + _wildify(syn) + r'\b', re.IGNORECASE))

    pats_na = [re.compile(r'\b' + _wildify(strip_accents(base)) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    pats_na += [re.compile(r'\b' + _wildify(strip_accents(syn)) + r'\b', re.IGNORECASE) for syn in syns]

    return {'canon': canon, 'patterns': pats, 'patterns_noacc': pats_na, 'synonyms': syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f"Thesaurus not found: {xlsx_path}")
    df = pd.read_excel(xlsx_path, engine="openpyxl")
    if column not in df.columns:
        raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')

    raw = df[column].dropna().astype(str).tolist()
    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', cell))

    seen, terms = set(), []
    for t in parts:
        t = (t or "").strip()
        if not t:
            continue
        low = t.lower()
        if len(low) == 1 or low in STOPWORDS:
            continue
        if t not in seen:
            seen.add(t)
            terms.append(t)
    print(f"[TH] Loaded {len(terms)} terms")
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))['canon'].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

def terms_matched_in_text(text: str, max_terms: int = 30) -> List[str]:
    hits = {}
    if not text:
        return []
    t_na = strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        cnt = max(a, b)
        if cnt > 0:
            hits[canon] = cnt
    ordered = [k for k,_ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0])))]
    return ordered[:max_terms]

# ============ DOCX/RTF SPLIT ============

_HEADING_NAME_PREFIXES = ('heading', 'nadpis','po-heading-id')
_HEADING_EXACT_NAMES   = {'title','subtitle','toc heading','obsah','po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or '').strip().lower()
    except: name = ''
    try: sid  = (getattr(p.style,'style_id','') or '').lower()
    except: sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name): return True
    if re.match(r'^heading\d+$', sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None:
                    buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,'r',encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None:
                    buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] RTF split failed {path}: {e}')
        return []

# ============ LLM LOADER ============

def _bitsandbytes_qconf():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

def _load_llm(model_id: str, tag: str):
    qconf = _bitsandbytes_qconf()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"

    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"  # batch-friendly

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=qconf,
        device_map=device_map,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

    pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tok,
            device_map=device_map,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            return_full_text=False,
            trust_remote_code=True,
            )
    return pipe, f"{tag}:{model_id}"

def load_primary_llm():
    return _load_llm(PRIMARY_MODEL_ID, "primary")

def load_secondary_llm():
    return _load_llm(SECONDARY_MODEL_ID, "secondary")

def load_fallback_llm():
    return _load_llm(FALLBACK_MODEL_ID, "fallback")

try:
    LLM_PIPE, USED_MODEL = load_primary_llm()
    print(f"[LLM] Loaded {USED_MODEL}")
except Exception as e:
    print(f"[WARN] Primary model failed: {e}")
    try:
        LLM_PIPE, USED_MODEL = load_secondary_llm()
        print(f"[LLM] Loaded {USED_MODEL}")
    except Exception as e2:
        print(f"[WARN] Secondary model failed: {e2}")
        LLM_PIPE, USED_MODEL = load_fallback_llm()
        print(f"[LLM] Loaded {USED_MODEL}")

# ============ RAG (len pre sekundár) ============

def _ensure_chroma_vectorstore() -> Optional[Chroma]:
    rag_enabled = (
        (USED_MODEL.startswith("secondary:") and ENABLE_RAG_SECONDARY) or
        (USED_MODEL.startswith("primary:")   and ENABLE_RAG_PRIMARY) or
        (USED_MODEL.startswith("fallback:")  and ENABLE_RAG_FALLBACK)
    )
    if not rag_enabled:
        print("[RAG] Disabled.")
        return None

    os.makedirs(RAG_PERSIST_DIR, exist_ok=True)
    embeddings = HuggingFaceEmbeddings(model_name=RAG_EMB_MODEL_NAME)

    try:
        vs = Chroma(
            collection_name=RAG_COLLECTION,
            persist_directory=RAG_PERSIST_DIR,
            embedding_function=embeddings,
        )
        _ = vs._collection.get(limit=1)
        print(f"[RAG] Enabled (top_k={RAG_TOP_K}).")
        return vs
    except Exception as e:
        print(f"[RAG] Vectorstore init failed, recreating: {e}")
        if os.path.isdir(RAG_PERSIST_DIR):
            shutil.rmtree(RAG_PERSIST_DIR, ignore_errors=True)
            os.makedirs(RAG_PERSIST_DIR, exist_ok=True)
        vs = Chroma(
            collection_name=RAG_COLLECTION,
            persist_directory=RAG_PERSIST_DIR,
            embedding_function=embeddings,
        )
        print("[RAG] Vectorstore ready (fresh).")
        return vs

VECTORSTORE = _ensure_chroma_vectorstore()

def rag_upsert(doc_id: str, texts: List[str], metas: List[dict]):
    if VECTORSTORE is None or not texts:
        return
    try:
        VECTORSTORE.add_texts(texts=texts, metadatas=metas, ids=[f"{doc_id}_{i}" for i in range(len(texts))])
    except Exception as e:
        print(f"[RAG] upsert failed: {e}")

def rag_retrieve(query: str, top_k: int = RAG_TOP_K) -> List[str]:
    if VECTORSTORE is None:
        return []
    try:
        docs = VECTORSTORE.similarity_search(query, k=top_k)
        return [d.page_content for d in docs]
    except Exception as e:
        print(f"[RAG] retrieve failed: {e}")
        return []

# ============ PROMPTY & REASONING ============

def truncate_text(s: str, max_chars: int = MAX_TEXT_CHARS) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars:
        return s
    return s[: max_chars//2] + "\n...\n" + s[- max_chars//2 :]

def _reasoning_wrap(task_prompt: str) -> str:
    # Inštrukcia na tiché rozvažovanie – vráť len finálny výstup
    return (
        "Premýšľaj potichu a dôkladne, ale do odpovede prosím nevypisuj postup ani úvahy.\n"
        "Vráť len finálny výsledok presne v požadovanom formáte.\n\n"
        + task_prompt
    )

def prompt_title(text: str, th_priority: List[str], th_sample: List[str], rag_ctx: List[str]) -> str:
    pri  = "\n".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = "\n".join(th_sample[:40])
    rag  = ("\n\nDOPLNKOVÝ KONTEXT (RAG):\n" + "\n---\n".join(rag_ctx)) if rag_ctx else ""
    core = (
        "ÚLOHA: Navrhni JEDEN presný a zrozumiteľný **slovenský** právny nadpis (3–12 slov) k textu nižšie.\n"
        "Požiadavky: vecný, bez bodky na konci, žiadne úvodzovky. Preferuj slovenské (nie české) výrazy.\n"
        "Vráť len samotný nadpis, nič iné.\n\n"
        f"TEZAURUS – PRIO (zhody v texte):\n{pri}\n\n"
        f"TEZAURUS – VÝBER:\n{samp}\n\n"
        f"TEXT:\n{text}{rag}\n\n"
        "NADPIS:"
    )
    return _reasoning_wrap(core)

def prompt_keyword(text: str, th_priority: List[str], th_sample: List[str], rag_ctx: List[str]) -> str:
    pri  = ", ".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = ", ".join(th_sample[:40])
    rag  = ("\n\nDOPLNKOVÝ KONTEXT (RAG):\n" + "\n---\n".join(rag_ctx)) if rag_ctx else ""
    core = (
        "ÚLOHA: Vyber JEDEN najvhodnejší **slovenský** právny pojem (1–4 slová) k nasledujúcemu textu.\n"
        "Preferuj pojmy z tezauru. Vráť len samotný pojem, bez vysvetlenia a bez bodky. Vyhni sa češtine.\n\n"
        f"TEZAURUS – PRIO: {pri}\n"
        f"TEZAURUS – VÝBER: {samp}\n\n"
        f"TEXT:\n{text}{rag}\n\n"
        "POJEM:"
    )
    return _reasoning_wrap(core)

def _normalize_output_item(item: Any) -> str:
    rec = item[0] if isinstance(item, list) else item
    raw = (
        rec.get("generated_text") if isinstance(rec, dict) else None
    ) or (
        rec.get("summary_text") if isinstance(rec, dict) else None
    ) or str(rec)
    txt = re.sub(r'^[\-\•\:\s]+', '', raw).strip()
    txt = re.sub(r'[\.，。…]+$', '', txt).strip()
    txt = txt.strip(' "„”')
    return txt.splitlines()[0].strip()

def _shape_group(outputs: Sequence[Any], n_inputs: int, nrs: int) -> List[List[Any]]:
    """
    Zoskup výstup pipeline do [n_inputs][<=nrs] kandidátov.
    - HF niekedy vráti List[List[Dict]] (už zoskupené),
        inokedy plochý List[Dict] dĺžky n_inputs*nrs.
    """
    if len(outputs) == n_inputs and isinstance(outputs[0], list):
        # už je to po položkách
        return [list(x) for x in outputs]
    # inak predpokladaj plochý zoznam
    grouped: List[List[Any]] = []
    cur = 0
    for _ in range(n_inputs):
        grouped.append(list(outputs[cur:cur+nrs]))
        cur += nrs
    return grouped

def safe_llm_call_batch(
        pipe, inputs, *, batch_size=1, num_return_sequences=1,
        max_new_tokens=None, temperature=None, top_p=None,
        max_retries=2, min_batch=1, min_new_tokens=32, min_samples=1, sleep_s=0.0
    ):
    """
    Vždy vráti: (outputs_list, used_batch_size, used_num_return_sequences, used_max_new_tokens).
    Ak všetko zlyhá, spraví per-input fallback (bs=1, nrs=1) a vráti zoznam dĺžky len(inputs).
    """
    import time

    bs  = max(1, int(batch_size))
    nrs = max(1, int(num_return_sequences))
    mnt = max_new_tokens

    def _call(_bs, _nrs, _mnt):
        kwargs = dict(batch_size=_bs, num_return_sequences=_nrs)
        if _mnt is not None: kwargs["max_new_tokens"] = _mnt
        if temperature is not None: kwargs["temperature"] = temperature
        if top_p is not None: kwargs["top_p"] = top_p
        return pipe(inputs, **kwargs)

    for attempt in range(max_retries + 1):
        try:
            out = _call(bs, nrs, mnt)
            return out, bs, nrs, mnt
        except RuntimeError as e:
            msg = str(e)
            # OOM/DSA – pokús sa adaptívne zmenšovať
            if ("CUDA out of memory" in msg) or ("device-side assert" in msg):
                changed = False
                if bs > min_batch:
                    bs = max(min_batch, bs // 2); changed = True
                elif nrs > min_samples:
                    nrs = max(min_samples, nrs // 2); changed = True
                elif mnt and mnt > min_new_tokens:
                    mnt = max(min_new_tokens, mnt // 2); changed = True

                if not changed:
                    break  # ideme na per-input fallback
                torch.cuda.empty_cache()
                time.sleep(sleep_s)
            else:
                # iné runtime chyby – skončíme per-input fallbackom
                break

    # ---- LAST-CHANCE: per-input fallback (bs=1, nrs=1, krátky výstup) ----
    outputs = []
    real_nrs = 1
    real_mnt = min_new_tokens if min_new_tokens else (mnt if mnt is not None else 32)
    real_mnt = max(16, min(64, real_mnt))

    for prompt in inputs:
        try:
            out = pipe(prompt, batch_size=1, num_return_sequences=1, max_new_tokens=real_mnt)
            outputs.append(out[0] if isinstance(out, list) else out)
        except Exception:
            outputs.append("")  # drž dĺžku a nechaj post-proc rozhodnúť

    return outputs, 1, real_nrs, real_mnt


def batched_generate_texts(prompts: List[str], batch_size: int = BATCH_SIZE) -> List[str]:
    outputs: List[str] = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        out, used_bs, used_nrs, _ = safe_llm_call_batch(
            LLM_PIPE, batch,
            batch_size=min(batch_size, len(batch)),
            num_return_sequences=1,
            max_new_tokens=MAX_NEW_TOKENS
        )
        for item in out:
            outputs.append(_normalize_output_item(item))
    if len(outputs) != len(prompts):
        print(f"[WARN] outputs={len(outputs)} != inputs={len(prompts)}")
    return outputs

# ---- Reasoning (self-consistency) ----

def _score_candidate(text: str, mode: str, prio_terms: List[str]) -> float:
    """Vyššie = lepšie. Vektor kritérií:
        1) Pokrytie tezauru (počty zhôd z prio_terms)
        2) Penalizácia češtiny
        3) Dĺžka: title 3–12 slov (ideál), keyword 1–4 slová (ideál)
    """
    t = text.strip()
    if not t:
        return -1e9
    # 1) pokrytie
    low = t.lower()
    cover = 0
    for term in prio_terms[:20]:
        trm = term.lower()
        if trm in low:
            cover += 1
    # 2) CZ penalizácia
    cz_hits = len(CZ_REGEX.findall(t)) if PENALIZE_CZ else 0
    cz_pen = -0.75 * cz_hits
    # 3) dĺžka
    nwords = len(re.findall(r"\w+", t))
    if mode == "title":
        if 3 <= nwords <= 12:
            len_score = 1.0
        else:
            len_score = -abs(nwords - 8) / 10.0
    else:
        if 1 <= nwords <= 4:
            len_score = 1.0
        else:
            len_score = -abs(nwords - 3) / 6.0
    return (1.0 * cover) + (0.6 * len_score) + (0.5 + cz_pen)

def batched_generate_texts_reasoned(
    jobs: List[Tuple[str, str, str, List[str], str]],
    batch_size: int = 2
) -> List[str]:
    """
    jobs: (mode, file, header, prio_terms, prompt)
    Vráti najlepšie odpovede po self-consistency pre každý vstup.
    """
    outputs: List[str] = []
    # priprav len prompty
    prompts = [j[-1] for j in jobs]
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        out, used_bs, used_nrs, used_mnt = safe_llm_call_batch(
            LLM_PIPE, batch,
            batch_size=min(batch_size, len(batch)),
            num_return_sequences=max(1, min(REASONING_SAMPLES, 6)),
            max_new_tokens=min(MAX_NEW_TOKENS, 128),
            temperature=REASONING_TEMP,
            top_p=REASONING_TOP_P,
            max_retries=2, min_batch=1, min_new_tokens=32, min_samples=1
        )
        # Zoskup podľa vstupov
        grouped = _shape_group(out, n_inputs=len(batch), nrs=max(1, min(REASONING_SAMPLES, 6)))
        if not grouped or any(g is None for g in grouped):
    # fallback na single-sample inference pre daný batch
            single_out, _, _, _ = safe_llm_call_batch(
                LLM_PIPE, batch, batch_size=1, num_return_sequences=1, max_new_tokens=64)
            grouped = _shape_group(single_out, n_inputs=len(batch), nrs=1)
        # Pre každý vstup vyber najlepšieho kandidáta
        for idx_in_batch, cand_list in enumerate(grouped):
            mode, f, header, prio_terms, _prompt = jobs[i + idx_in_batch]
            texts = [_normalize_output_item(c) for c in cand_list if c is not None]
            if not texts:
                outputs.append("")
                continue
            # Scoring
            scored = [(txt, _score_candidate(txt, mode, prio_terms)) for txt in texts]
            best = max(scored, key=lambda x: x[1])[0]
            outputs.append(best)
            
    if len(outputs) != len(prompts):
        print(f"[WARN] reasoned outputs={len(outputs)} != inputs={len(prompts)}")
    return outputs

def penalize_czech(s: str) -> str:
    if not PENALIZE_CZ or not s:
        return s
    s2 = s
    s2 = re.sub(r"\břízení\b", "konanie", s2, flags=re.IGNORECASE)
    s2 = re.sub(r"\bustanovení\b", "ustanovenie", s2, flags=re.IGNORECASE)
    s2 = re.sub(r"\bsprávní\b", "správne", s2, flags=re.IGNORECASE)
    return s2

# ============ DRIVER ============

def process_all(input_dir=INPUT_DIR):
    rows = []
    files = sorted(os.listdir(input_dir), key=str.lower)

    # RAG aktívny?
    rag_active = (
        (USED_MODEL.startswith("secondary:") and ENABLE_RAG_SECONDARY) or
        (USED_MODEL.startswith("primary:")   and ENABLE_RAG_PRIMARY) or
        (USED_MODEL.startswith("fallback:")  and ENABLE_RAG_FALLBACK)
    )

    # Reasoning aktívny?
    reasoning_active = (
        (USED_MODEL.startswith("primary:")   and REASONING_FOR_PRIMARY) or
        (USED_MODEL.startswith("secondary:") and REASONING_FOR_SECONDARY) or
        (USED_MODEL.startswith("fallback:")  and REASONING_FOR_FALLBACK)
    )

    # 1) Build jobs
    jobs = []  # (mode, file, header, prio_terms, prompt)
    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith(".docx"):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith(".rtf"):
            secs = split_rtf_by_headers(p)
        else:
            continue
        if not secs:
            print(f"No text in {f}")
            continue

        # RAG upsert pre sekcie
        if rag_active and VECTORSTORE is not None:
            section_texts, metas = [], []
            for idx, (header, text) in enumerate(secs):
                if header == "__DOCUMENT__":
                    continue
                if text.strip():
                    section_texts.append(text.strip())
                    metas.append({"file": f, "header": header, "idx": idx})
            if section_texts:
                rag_upsert(os.path.splitext(f)[0], section_texts, metas)

        for header, text in secs:
            if not text.strip():
                continue
            short = truncate_text(text, MAX_TEXT_CHARS)
            prio_terms = terms_matched_in_text(text, max_terms=30)
            sample_terms = prio_terms if prio_terms else CANON_TERMS[:40]
            rag_ctx = rag_retrieve(short, top_k=RAG_TOP_K) if rag_active else []

            if MODE == "title":
                prompt = prompt_title(short, prio_terms, sample_terms, rag_ctx)
                jobs.append(("title", f, header, prio_terms, prompt))
            else:
                prompt = prompt_keyword(short, prio_terms, sample_terms, rag_ctx)
                jobs.append(("keyword", f, header, prio_terms, prompt))

        print(f"Queued {f} with {len(secs)} sections.")

    if not jobs:
        print("No jobs queued.")
        return pd.DataFrame([])

    # 2) Inference (reasoning vs. single)
    if reasoning_active and REASONING_SAMPLES >= 2:
        gens = batched_generate_texts_reasoned(jobs, batch_size=2)
    else:
        prompts = [j[-1] for j in jobs]
        gens = batched_generate_texts(prompts, batch_size=BATCH_SIZE)

    # 3) Compose results (+ jemná CZ->SK korekcia)
    for (mode, f, header, prio_terms, _), gen in zip(jobs, gens):
        gen = penalize_czech(gen)
        if mode == "title":
            rows.append({
                "file": f,
                "header": header,
                "suggested_title": gen,
                "method": f"{'RAG+' if rag_active else ''}{'reasoned-' if reasoning_active else ''}llm ({USED_MODEL})",
                "priority_terms": "; ".join(prio_terms[:20])
            })
        else:
            rows.append({
                "file": f,
                "header": header,
                "top_keyword": gen,
                "method": f"{'RAG+' if rag_active else ''}{'reasoned-' if reasoning_active else ''}llm ({USED_MODEL})",
                "priority_terms": "; ".join(prio_terms[:20])
            })

    today = datetime.today().strftime("%Y-%m-%d")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if MODE == "title":
        df = pd.DataFrame(rows, columns=["file","header","suggested_title","method","priority_terms"])
        stub = "titles"
    else:
        df = pd.DataFrame(rows, columns=["file","header","top_keyword","method","priority_terms"])
        stub = "keywords"

    csv_path = os.path.join(OUTPUT_DIR, f"{stub}_{today}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {csv_path}")
    return df

# ------------------------------
# MAIN
# ------------------------------
if __name__ == "__main__":
    df = process_all(INPUT_DIR)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))


[STOP] Loaded 76 stopwords
[TH] Loaded 3000 terms
[WARN] Primary model failed: name 'PRIMARY_MODEL_ID' is not defined


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[LLM] Loaded secondary:Qwen/Qwen2.5-3B-Instruct


C:\Users\nyx3t\AppData\Local\Temp\ipykernel_17588\3113612493.py:347: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=RAG_EMB_MODEL_NAME)


[RAG] Enabled (top_k=5).
Queued 117694.docx with 8 sections.
Queued 117695.docx with 2 sections.
Queued 117696.docx with 11 sections.
Queued 117697.docx with 15 sections.
Queued 117698.docx with 8 sections.
Queued 117699.docx with 18 sections.
Queued 117700.docx with 1 sections.
Queued 117701.docx with 6 sections.
Queued 117702.docx with 12 sections.
Queued 117703.docx with 10 sections.


KeyboardInterrupt: 